In [ ]:
#Stepmotor, have motor pins be on GPIO 17,18,19,20.
import RPi.GPIO as GPIO
from RpiMotorLib.RpiMotorLib import BYJMotor
from matplotlib import pyplot as py
import math
import threading
import time

GPIO.setmode(GPIO.BCM)

pins = [17,18,27,22]
GPIO.setup(pins,GPIO.OUT)
motor = BYJMotor("stepper", "28BYJ48")
x = []
y = []
displacement = 0.0
start_time = time.time()
running = True

def bglogging():
    global displacement, running
    while running:
        elapsed = time.time()-start_time
        x.append(elapsed)
        y.append(displacement)
        time.sleep(0.1)

def get_valid_input(prompt, value_type=float):
    while True:
        try:
            return value_type(input(prompt))
        except ValueError:
            print(f"Invalid input. Please enter a {value_type.__name__}.")

def loop():
    global displacement
    while True:
        rev = get_valid_input("Enter number of revolutions: ")
        steps = rev * 512
        speed = get_valid_input("Enter angular speed in radians per seconds (max 1.5): ", float)
        #need to correct speed conversion
        delay = (2* math.pi)/(512*speed)
        direction = -1
        while direction not in [0, 1]:
            direction = get_valid_input("Enter direction (0 for CW, 1 for CCW): ", int)
        
        is_ccw = (direction == 1)

        print(f"Running: {rev} revolutions, {speed} rad/s, CCW={is_ccw}")

        rad_per_step = (2*math.pi)/512*speed
        step_value = rad_per_step if is_ccw else -rad_per_step
        
        displacement += (step_value * steps)
            
        
        # Execute movement
        motor.motor_run(pins, delay, steps, is_ccw, False, "half", 0.05)

if __name__ == "__main__":
    try:
        log_thread = threading.Thread(target=bglogging, daemon=True)
        log_thread.start()
        loop()
    except KeyboardInterrupt:
        running = False
        plt.figure(figsize=(10,5))
        py.plot(x,y)
        plt.title("Angular Displacement as a function of time")
        plt.xlabel("Time (seconds)")
        plt.ylabel("Displacemene (radians)")
        plt.grid(True)
        plt.show()
        GPIO.output(pins, GPIO.LOW)
        GPIO.cleanup()

Enter number of revolutions:  1
Enter angular speed in radians per seconds (max 1.5):  2
Enter direction (0 for CW, 1 for CCW):  0


Running: 1.0 revolutions, 2.0 rad/s, CCW=False
User Keyboard Interrupt : RpiMotorLib: 
